# Práctica de RAG — narrativa completa (naive desde cero → LangChain hybrid + reranker)

<a href="https://colab.research.google.com/github/jorgeroa/ia-utn-frsf/blob/main/clase03/notebooks/rag/practica_completa.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> **Objetivo:** construir un sistema RAG sobre el programa real de la Cátedra IA UTN-FRSF 2026, pasarlo por un benchmark de 7 queries diversas, y comparar **RAG implementado a mano** (para entender qué pasa adentro) contra **RAG con LangChain** (para entender qué usás en producción).
>
> **Por qué este corpus:** el programa de la cátedra (docentes, ordenanzas, fechas, umbrales de aprobación) contiene datos **imposibles de adivinar** para el LLM. El contraste RAG-vs-LLM puro es dramático y mensurable.
>
> **Estructura:**
> - **Parte 1** — Setup + motivación (LLM puro vs RAG hardcoded).
> - **Parte 2** — Pipeline RAG naive completo a mano (corpus → chunking → ChromaDB → retrieval → augmentation → generation).
> - **Parte 3** — Benchmark sobre el corpus + correr naive con side-by-side y visualización.
> - **Parte 4** — Subir a un framework: LangChain con `EnsembleRetriever` (hybrid con RRF) y `ContextualCompressionRetriever` (reranking con cross-encoder). Mismo benchmark, comparación final.
>
> **Tiempo:** ~75-90 min de hands-on en total (se puede partir en sesiones).


# Parte 1 — Setup + motivación

Antes de meter retrieval automático, veamos qué pasa cuando le preguntamos al LLM cosas que **no puede saber**.


## 0. Setup

Instalá dependencias y configurá la API key de Groq.


In [ ]:
!pip install -q sentence-transformers chromadb rank_bm25 groq numpy pandas matplotlib


In [ ]:
# OPCIONAL — sólo para correr en local. En Colab esta celda se autodetecta y se omite
# (en Colab configurá GROQ_API_KEY desde "Secrets" en la barra lateral izquierda).
#
# Si vos corrés esta notebook localmente con tu venv:
#   1. Poné GROQ_API_KEY=tu-api-key en un archivo .env en la raíz del repo
#   2. Esta celda lo carga automáticamente

try:
    import google.colab  # noqa: F401
    print('ℹ️  Colab detectado — saltando .env (usá Colab Secrets para GROQ_API_KEY).')
except ImportError:
    try:
        from dotenv import load_dotenv, find_dotenv
    except ImportError:
        import subprocess, sys
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'python-dotenv'])
        from dotenv import load_dotenv, find_dotenv
    env_path = find_dotenv(usecwd=True)
    if env_path:
        load_dotenv(env_path)
        print(f'✓ .env cargado desde {env_path}')
    else:
        print('ℹ️  No se encontró .env. Asegurate de exportar GROQ_API_KEY como env var antes de lanzar jupyter.')


In [ ]:
import os

try:
    from google.colab import userdata
    if 'GROQ_API_KEY' not in os.environ:
        try:
            os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
        except Exception:
            pass
except ImportError:
    pass

assert os.environ.get('GROQ_API_KEY'), (
    'Falta GROQ_API_KEY. Configurala en Colab Secrets o como env var.'
)
print('✓ GROQ_API_KEY configurada')


In [ ]:
from groq import Groq

_groq_client = Groq(api_key=os.environ['GROQ_API_KEY'])


def llamar_llm(messages, model='llama-3.1-8b-instant', temperature=0.3, max_tokens=1024):
    """Wrapper unificado para llamar Groq — el mismo de Clase 2."""
    resp = _groq_client.chat.completions.create(
        model=model, messages=messages,
        temperature=temperature, max_tokens=max_tokens,
    )
    return resp.choices[0].message.content


print(llamar_llm([{'role': 'user', 'content': 'Decime "ok" si me podés escuchar'}]))


In [ ]:
from sentence_transformers import SentenceTransformer

modelo_emb = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print(f'✓ Modelo de embeddings: {modelo_emb.get_sentence_embedding_dimension()} dims')


## 1. Motivación — el LLM no puede saber esto

**Pregunta concreta:** *"¿Cuándo es el parcial de Inteligencia Artificial 2026 de la UTN-FRSF y qué tema cubre?"*

Esto **no está en el training del LLM**. Vamos a ver:
1. Qué responde el LLM puro — muy probablemente **inventa con confianza**.
2. Qué responde con contexto manual (RAG hardcoded simulado).

Después automatizamos la búsqueda del contexto.


In [ ]:
pregunta = '¿Cuándo es el parcial de Inteligencia Artificial 2026 de la UTN-FRSF y qué tema cubre?'

# Hechos verificables (sacamos "06/04" porque el LLM va a escribir "6 de abril" en prosa)
hechos_verdaderos = ['6 de abril', 'Búsqueda en espacio de estado', 'campus virtual']


def score_keywords(respuesta, keywords):
    return sum(1 for k in keywords if k.lower() in respuesta.lower())


# ── 1) LLM PURO (sin contexto) ──
resp_sin = llamar_llm([
    {'role': 'system', 'content':
        'Respondé en español, máximo 4 oraciones. Sé directo: si tenés información, dala; '
        'si no tenés, decilo breve.'},
    {'role': 'user', 'content': pregunta},
], temperature=0.3)
score_sin = score_keywords(resp_sin, hechos_verdaderos)

print('╔════════════════════════════════════════════════════════════╗')
print('║   LLM PURO (sin RAG)                                       ║')
print('╠════════════════════════════════════════════════════════════╣')
print(resp_sin)
print()
print(f'   Score de hechos correctos: {score_sin}/{len(hechos_verdaderos)}')
print('   ⚠️  Dos resultados posibles:')
print('       (a) admite no saber → score 0 pero el usuario queda sin respuesta')
print('       (b) inventa fechas o temas plausibles → score 0 y el usuario es engañado')
print('   En ambos casos: el LLM puro no resuelve el problema del usuario.')
print('╚════════════════════════════════════════════════════════════╝')

# ── 2) CON contexto manual explícito ──
# Clave: el contexto debe ser INEQUÍVOCO sobre a qué materia/año se refiere.
contexto_manual = (
    'Contexto sobre la cátedra Inteligencia Artificial — UTN-FRSF — 1er cuatrimestre 2026: '
    'El parcial de la materia tiene fecha 6 de abril de 2026 y cubre el tema '
    '"Búsqueda en espacio de estado". Se rinde en clase usando el campus virtual.'
)
resp_con = llamar_llm([
    {'role': 'system', 'content':
        'Sos un asistente de la cátedra. Respondé la pregunta usando la información del '
        'contexto provisto. Sé concreto, citá los datos exactos. Máximo 4 oraciones.'},
    {'role': 'user', 'content': f'Contexto:\n{contexto_manual}\n\nPregunta: {pregunta}'},
], temperature=0.3)
score_con = score_keywords(resp_con, hechos_verdaderos)

print()
print('╔════════════════════════════════════════════════════════════╗')
print('║   CON contexto (RAG hardcoded)                             ║')
print('╠════════════════════════════════════════════════════════════╣')
print(resp_con)
print()
print(f'   ✓ Score de hechos correctos: {score_con}/{len(hechos_verdaderos)}')
print('   ✓ Información trazable al contexto provisto.')
print('╚════════════════════════════════════════════════════════════╝')

print()
print(f'💡 Diferencia visible: {score_con} vs {score_sin}. El RAG no es "más inteligente" —')
print('   tiene acceso a información que el LLM literalmente no puede saber.')
print('   Ahora vamos a automatizar la búsqueda del contexto.')


## 1b. ¿Y si el LLM **igual responde** sin saber? — la alucinación silenciosa

El caso del parcial el LLM lo manejó bien (admitió no saber). Pero hay preguntas donde el LLM **no se contiene** y responde **con confianza pero mal**.

Probemos con una pregunta procedural sobre las prácticas:


In [ ]:
pregunta_b = '¿Cuántos integrantes pueden tener los grupos de trabajos prácticos en la cátedra Inteligencia Artificial de UTN-FRSF? ¿Y cómo se inscriben los grupos?'

# Ground truth de nuestro corpus: 2 o 3 integrantes (1 < n ≤ 3), inscripción por email
hechos_b = ['2', '3', 'email']

resp_b_sin = llamar_llm([
    {'role': 'system', 'content':
        'Respondé en español, máximo 4 oraciones. Sé directo y concreto. Si tenés información, dala.'},
    {'role': 'user', 'content': pregunta_b},
], temperature=0.3)
score_b_sin = score_keywords(resp_b_sin, hechos_b)

print('╔════════════════════════════════════════════════════════════╗')
print('║   LLM PURO — pregunta procedural sin info real             ║')
print('╠════════════════════════════════════════════════════════════╣')
print(resp_b_sin)
print()
print(f'   Score: {score_b_sin}/{len(hechos_b)}')
print('   ⚠️  Fijate si el LLM:')
print('       - Inventó un rango de integrantes (común: 3-5 o "depende de la cátedra")')
print('       - Inventó un método de inscripción (campus, sistema, formulario)')
print('   Ese es el problema MÁS PELIGROSO: una respuesta plausible pero falsa.')
print('╚════════════════════════════════════════════════════════════╝')

# Con contexto manual
contexto_b = (
    'Contexto sobre la cátedra Inteligencia Artificial — UTN-FRSF — 2026: '
    'Los trabajos prácticos y las actividades en clase se realizan en forma grupal. '
    'Los grupos están formados por 2 o 3 integrantes (1 < nro-integrantes ≤ 3). '
    'Cada grupo debe enviar por email su formación indicando nombre, apellido y '
    'dirección de email de todos los integrantes.'
)
resp_b_con = llamar_llm([
    {'role': 'system', 'content':
        'Sos un asistente de la cátedra. Respondé usando el contexto. Sé concreto, citá los datos exactos. Máximo 4 oraciones.'},
    {'role': 'user', 'content': f'Contexto:\n{contexto_b}\n\nPregunta: {pregunta_b}'},
], temperature=0.3)
score_b_con = score_keywords(resp_b_con, hechos_b)

print()
print('╔════════════════════════════════════════════════════════════╗')
print('║   CON contexto                                              ║')
print('╠════════════════════════════════════════════════════════════╣')
print(resp_b_con)
print()
print(f'   ✓ Score: {score_b_con}/{len(hechos_b)}')
print('╚════════════════════════════════════════════════════════════╝')

print()
print(f'💡 La pregunta es: ¿el LLM puro **inventó** algo en su respuesta?')
print('   Si la respuesta contiene un número distinto a 2 o 3, o menciona inscripción')
print('   por sistema/campus en vez de email → ESO ES ALUCINACIÓN.')
print('   El RAG con contexto responde con los datos exactos del corpus.')


## 1c. Reflexión — la alucinación moderna es **más sutil** (y más peligrosa)

Si corriste la celda anterior con un modelo moderno (Llama 3.1/3.3, GPT-4+, Claude), es posible que hayas visto un patrón parecido a este:

> *"No tengo información actualizada sobre la estructura específica... **Sin embargo**, en general los grupos de trabajo pueden tener entre 3 a 6 integrantes... Para inscribirse, consultar el sitio web... presentar solicitud en secretaría..."*

### ¿Qué podría haber pasado?

| Comportamiento posible | Análisis |
|---|---|
| El modelo empieza con un disclaimer ("no tengo info específica") | ✓ Calibración correcta — sabe que no sabe |
| **Pero igual responde** con info "general" | ✗ Eso **sería** alucinación, con disfraz |
| Da números específicos (ej. 3-6) | ✗ La verdad es 2-3 |
| Da método de inscripción específico (web / secretaría) | ✗ La verdad es email |
| Sugiere verificar con la universidad | ✓ Buen consejo pero llega tarde |

### La alucinación "descarada" de 2023 ya casi no existe

Hasta 2023 los LLMs alucinaban **sin pudor** ("el parcial es el 15 de abril y cubre redes neuronales convolucionales" — todo inventado, sin disclaimer). Era fácil de detectar visualmente.

Con RLHF (Clase 2) los modelos aprendieron a **advertir su incertidumbre primero, inventar igual después**. Eso es **más peligroso** porque:

1. El disclaimer da una sensación falsa de calibración ("el modelo me avisa cuando no sabe").
2. La info inventada cuela igual, escondida tras el "en general".
3. El usuario no puede distinguir "info general verdadera" de "info general inventada".

### Cuándo el score podría mentir

Supongamos que el LLM respondió algo como *"entre 3 y 6 integrantes"*. El `score_keywords` matchearía el `'3'` aunque la respuesta **sería incorrecta** (la verdad es 2 o 3, no 3 a 6). En ese caso el score `2/3` estaría diciendo "67% correcto" cuando en realidad fue 0% correcto.

Este sería exactamente el problema que motiva **LLM-as-judge** (Clase 3b): el match por substring puede engañar. Un juez LLM podría leer la respuesta, distinguir "2 o 3" de "3 a 6", y darle el score real (0).

### El takeaway

Demostrar la "alucinación dramática" en clase con modelos modernos requiere preguntas más sutiles — o aceptar que el contraste viene de **info general inventada** + **disclaimer aparentemente honesto**. En cualquier caso, **el RAG resuelve el problema de raíz**: el modelo deja de adivinar porque tiene la respuesta exacta en el contexto.


# Parte 2 — Pipeline RAG naive completo

Reemplazamos el "contexto manual" del paso anterior por un sistema automático:
**corpus → chunking → embeddings → vector store → retrieval → augmentation → generation**.


## 2. El corpus — programa real de la Cátedra IA 2026

5 documentos sobre el programa de la materia. Los datos son específicos: docentes con cargo, fechas, ordenanzas, umbrales, aula. **El LLM puro no puede inventar nada de esto correctamente.**


In [ ]:
# ── Corpus: Cátedra Inteligencia Artificial — UTN-FRSF, 1er cuatrimestre 2026 ──

DOCUMENTOS = [
    {
        "id": "doc_identidad",
        "titulo": "Identidad y modalidad de la cátedra",
        "contenido": (
            "La asignatura Inteligencia Artificial pertenece al 5to año de la carrera "
            "Ingeniería en Sistemas de Información (ISI) de la Universidad Tecnológica "
            "Nacional, Facultad Regional Santa Fe (UTN-FRSF), y se dicta durante el "
            "1er cuatrimestre del ciclo 2026. La cátedra está radicada en el CIDISI "
            "(Centro de Investigación de Desarrollo e Innovación en Sistemas de "
            "Información). Los docentes a cargo son la Profesora Titular Milagros "
            "Gutiérrez y el Profesor Adjunto Jorge Roa. La cátedra no cuenta con "
            "auxiliares ni ayudantes alumnos en esta cohorte. La materia se dicta "
            "de manera presencial los días lunes de 18:00 a 21:00 horas en el "
            "Laboratorio 1 de la facultad. La modalidad de cursado es por proyecto, "
            "con foco en aplicación práctica más que en exposición magistral. La "
            "cátedra opera bajo el marco normativo de la Ordenanza 1877, que "
            "establece las competencias específicas (CE) y genéricas (CG) requeridas "
            "para todas las ingenierías de la UTN. Las competencias específicas "
            "incluyen CE1.1 (especificar y desarrollar sistemas de información), "
            "CE1.3 (desarrollar software para soluciones informáticas), CE4.1 "
            "(certificar funcionamiento de sistemas) y CE5.1 (dirigir implementación "
            "de sistemas). Las competencias genéricas incluyen CG4 (uso de técnicas "
            "y herramientas), CG5 (generar desarrollos tecnológicos), CG8 (ética y "
            "responsabilidad profesional) y CG9 (aprendizaje continuo)."
        ),
    },
    {
        "id": "doc_evaluacion",
        "titulo": "Regularidad, promoción directa y asistencia",
        "contenido": (
            "Para regularizar el cursado de Inteligencia Artificial el estudiante "
            "debe: (a) aprobar los dos trabajos prácticos con un puntaje igual o "
            "superior a 4, (b) aprobar el parcial con un puntaje igual o superior "
            "a 4, y (c) realizar las actividades de evaluación continua y participar "
            "en el debate sobre ética en IA. Adicionalmente se requiere el 80 por "
            "ciento de asistencia. Las condiciones para la promoción directa son "
            "las mismas, pero los puntajes mínimos suben a 6 tanto en los dos "
            "trabajos prácticos como en el parcial; también se requiere haber "
            "realizado las actividades de evaluación continua y haber participado "
            "del debate, manteniendo el 80 por ciento de asistencia. La nota final "
            "es el promedio de las notas de los trabajos prácticos, las evaluaciones "
            "continuas y el parcial. Los alumnos que demuestren niveles mínimos y "
            "básicos de aprendizaje pero no hayan alcanzado la aprobación directa "
            "rendirán un examen final escrito que abarca todos los temas del "
            "programa."
        ),
    },
    {
        "id": "doc_tps",
        "titulo": "Trabajos prácticos y modalidad de agrupamiento",
        "contenido": (
            "La materia tiene dos trabajos prácticos. El TP1 consiste en diseñar un "
            "clasificador neuronal y se realiza en conjunto con la cátedra de "
            "Ciencia de Datos. El TP2 consiste en diseñar un agente con temática "
            "libre y se divide en tres etapas: la primera etapa es la definición "
            "del agente y los objetivos de diseño; la segunda etapa es la "
            "definición de percepciones y acciones del agente más la descripción "
            "del ambiente donde actúa, junto con implementaciones parciales; la "
            "tercera etapa es la entrega final con código, presentación e informe. "
            "Los trabajos prácticos y las actividades en clase se realizan en forma "
            "grupal. Los grupos están formados por 2 o 3 integrantes — es decir, "
            "1 menor estricto que nro-integrantes menor o igual que 3. Cada grupo "
            "debe enviar por email su formación indicando nombre, apellido y "
            "dirección de email de todos los integrantes. Los aspectos a evaluar "
            "en cada TP son: el informe técnico y la presentación, la definición y "
            "alcance de los objetivos del TP, la presentación de la solución de "
            "software, la cantidad y calidad de las herramientas que se apliquen, "
            "la calidad y cantidad de pruebas de validación, y la expresividad en "
            "los coloquios de presentación de los prácticos."
        ),
    },
    {
        "id": "doc_cronograma",
        "titulo": "Programación de actividades 1er cuatrimestre 2026",
        "contenido": (
            "La programación de actividades para el 1er cuatrimestre 2026 es la "
            "siguiente. El debate en equipos sobre ética en IA se realiza el 16 de "
            "marzo. El parcial tiene fecha 6 de abril y cubre el tema Búsqueda en "
            "espacio de estado, y se rinde en clase usando el campus virtual. El "
            "TP1 sobre el desarrollo del modelo neuronal tiene fecha de entrega "
            "el 18 de mayo. El TP2 se desarrolla en tres etapas con entregas "
            "sucesivas: la primera etapa, definición del agente y objetivos de "
            "diseño, vence el 8 de junio; la segunda etapa, definición de "
            "percepciones y acciones del agente con descripción del ambiente e "
            "implementaciones parciales, vence el 22 de junio; la entrega final "
            "con código, presentación e informe vence el 29 de junio. Las clases "
            "del cuatrimestre se dictan los lunes de 18 a 21 horas en el "
            "Laboratorio 1."
        ),
    },
    {
        "id": "doc_recursos",
        "titulo": "Bibliografía, software y canal de comunicación",
        "contenido": (
            "La bibliografía principal de la materia incluye: Inteligencia "
            "Artificial Un enfoque moderno de Russell y Norvig, disponible en "
            "aima.cs.berkeley.edu; Inteligencia Artificial Segunda Edición de Rich "
            "y Knight; Inteligencia Artificial Una nueva síntesis de Nilsson; "
            "MultiAgent Systems de Wooldridge; y los apuntes de clase elaborados "
            "por la cátedra. El software utilizado incluye Python como lenguaje "
            "principal, notebooks Jupyter para los desarrollos y Google Colab "
            "como entorno de ejecución en la nube. El canal de comunicación oficial "
            "entre estudiantes y docentes es el campus virtual de la FRSF, "
            "disponible en https://campusvirtual.frsf.utn.edu.ar/, donde se "
            "publican los materiales, se administran los cuestionarios de "
            "evaluación continua y se entregan los trabajos prácticos. Los "
            "objetivos pedagógicos generales de la cátedra son: gestionar proyectos "
            "de construcción de sistemas inteligentes, reconocer estrategias de "
            "creación de sistemas inteligentes, resolver problemas de "
            "representación del conocimiento y razonamiento en ambientes "
            "deterministas y bajo incertidumbre, y evaluar e implementar modelos "
            "de aprendizaje automático para resolver problemas."
        ),
    },
]

print(f'✓ {len(DOCUMENTOS)} documentos de la cátedra IA cargados')
for doc in DOCUMENTOS:
    print(f'  - {doc["titulo"]} ({len(doc["contenido"])} chars)')


## 3. Chunking

Partimos cada documento en **oraciones** (split por `. ! ?`). Es lo más simple y funciona bien para textos de reglamento donde cada oración es autocontenida.


In [ ]:
import re


def chunk_por_oracion(texto):
    """Chunking por oración (split en punto/exclamación/interrogación + espacio)."""
    oraciones = re.split(r'(?<=[.!?])\s+', texto)
    return [o.strip() for o in oraciones if o.strip()]


all_chunks = []
all_ids = []
all_metadatas = []
for doc in DOCUMENTOS:
    chunks = chunk_por_oracion(doc['contenido'])
    for i, chunk in enumerate(chunks):
        all_chunks.append(chunk)
        all_ids.append(f'{doc["id"]}_chunk_{i}')
        all_metadatas.append({
            'titulo': doc['titulo'],
            'doc_id': doc['id'],
            'chunk_index': i,
        })

print(f'✓ {len(all_chunks)} chunks (oraciones) listas para indexar')
print(f'\nEjemplos:')
for chunk in all_chunks[:3]:
    print(f'  - "{chunk[:80]}..." ({len(chunk)} chars)')


## 4. Embeddings + ChromaDB (indexing)

Generamos un embedding por chunk y los almacenamos en ChromaDB in-memory.


In [ ]:
import chromadb

client = chromadb.Client()
try:
    client.delete_collection('clase3_docs')
except Exception:
    pass

collection = client.create_collection(
    name='clase3_docs',
    metadata={'hnsw:space': 'cosine'},
)

all_embeddings = modelo_emb.encode(all_chunks).tolist()

collection.add(
    documents=all_chunks,
    embeddings=all_embeddings,
    metadatas=all_metadatas,
    ids=all_ids,
)
print(f'✓ Indexados {collection.count()} chunks en ChromaDB')


## 5. Retrieval

Dada una query, buscamos los top-k chunks más similares por coseno.


In [ ]:
def buscar_chunks(query, n_results=3):
    """Retrieval denso: top-k por similitud coseno."""
    query_embedding = modelo_emb.encode([query]).tolist()
    return collection.query(
        query_embeddings=query_embedding,
        n_results=n_results,
        include=['documents', 'metadatas', 'distances'],
    )


query = '¿Quiénes son los docentes de la cátedra?'
results = buscar_chunks(query, n_results=3)
print(f'Query: "{query}"\n\nTop 3 chunks:')
for i in range(len(results['documents'][0])):
    titulo = results['metadatas'][0][i]['titulo']
    sim = 1 - results['distances'][0][i]
    print(f'  #{i+1} (sim {sim:.3f}) [{titulo}]: "{results["documents"][0][i][:80]}..."')


## 6. RAG end-to-end — `rag_query`

System prompt fuerza al LLM a **basarse SOLO en el contexto** y **citar fuentes**.


In [ ]:
SYSTEM_RAG = """Sos un asistente de la cátedra Inteligencia Artificial de UTN-FRSF
que responde preguntas basándose ÚNICAMENTE en el contexto proporcionado.

Reglas:
- Usá SOLO la información del contexto. Nunca inventes fechas, nombres o números.
- Si el contexto no tiene información suficiente, decí "No tengo información suficiente en los documentos proporcionados."
- Citá la fuente entre corchetes cuando sea posible, ej: [Cronograma 2026].
- Respondé en español, conciso (máximo 5 oraciones)."""


def rag_query(pregunta, n_chunks=3, verbose=False):
    """Pipeline RAG completo: retrieval → augmentation → generation."""
    results = buscar_chunks(pregunta, n_results=n_chunks)

    contexto_partes = []
    for i in range(len(results['documents'][0])):
        doc = results['documents'][0][i]
        titulo = results['metadatas'][0][i]['titulo']
        contexto_partes.append(f'[{i+1}] ({titulo}): {doc}')
    contexto = '\n\n'.join(contexto_partes)

    messages = [
        {'role': 'system', 'content': SYSTEM_RAG},
        {'role': 'user', 'content': f'Contexto:\n{contexto}\n\nPregunta: {pregunta}'},
    ]
    respuesta = llamar_llm(messages, temperature=0.3)

    if verbose:
        print(f'Pregunta: {pregunta}\n')
        print(f'📎 Chunks recuperados:')
        for parte in contexto_partes:
            print(f'  {parte[:100]}...')
        print(f'\n💬 Respuesta:\n{respuesta}')

    return respuesta, results


respuesta, _ = rag_query('¿Cuáles son las fechas de entrega del TP2?', verbose=True)


# Parte 3 — Benchmark + correr RAG Naive

Definimos un benchmark de 7 queries sobre el programa, y lo corremos en paralelo por **LLM puro** y **RAG naive** para ver el contraste en datos.


## 7. El benchmark — 7 queries sobre el programa de la cátedra

Diseño deliberado para cubrir 4 tipos de query (fácil / ambigua-sigla / ambigua-concepto / multi-hop / edge case). Cada query golpea **datos específicos** que el LLM puro **no puede inventar correctamente**.


In [ ]:
BENCHMARK = [
    # ── Fáciles ──────────────────────────────────────────────────
    {
        'id': 'easy-1', 'tipo': 'easy',
        'pregunta': '¿Quiénes son los docentes de la cátedra Inteligencia Artificial en UTN-FRSF y qué cargo tiene cada uno?',
        'expected_keywords': ['Milagros Gutiérrez', 'Jorge Roa', 'Titular', 'Adjunto'],
        'expected_doc': 'doc_identidad',
    },
    {
        'id': 'easy-2', 'tipo': 'easy',
        'pregunta': '¿Qué día, horario y aula se dicta la materia?',
        'expected_keywords': ['lunes', '18', '21', 'Laboratorio 1', 'presencial'],
        'expected_doc': 'doc_identidad',
    },
    # ── Ambigua: sigla local ─────────────────────────────────────
    {
        'id': 'amb-1', 'tipo': 'ambigua-sigla',
        'pregunta': '¿Qué es el CIDISI y qué establece la Ordenanza 1877?',
        'expected_keywords': ['Centro de Investigación', 'Sistemas de Información', 'competencias', 'CE', 'CG'],
        'expected_doc': 'doc_identidad',
    },
    # ── Ambigua: concepto amplio ─────────────────────────────────
    {
        'id': 'amb-2', 'tipo': 'ambigua-concepto',
        'pregunta': '¿Cómo está organizado el sistema de evaluación de la materia?',
        'expected_keywords': ['trabajos prácticos', 'parcial', 'evaluación continua', 'debate', 'asistencia'],
        'expected_doc': 'doc_evaluacion',
    },
    # ── Multi-hop ────────────────────────────────────────────────
    {
        'id': 'multi-1', 'tipo': 'multi-hop',
        'pregunta': '¿Qué necesito hacer para promocionar la materia y qué fechas tengo que cumplir?',
        'expected_keywords': ['6', 'trabajos prácticos', 'parcial', '80', 'asistencia', '16', 'marzo', 'abril', '18', 'mayo', '29', 'junio'],
        'expected_docs': ['doc_evaluacion', 'doc_cronograma'],
    },
    {
        'id': 'multi-2', 'tipo': 'multi-hop',
        'pregunta': '¿Cuál es la diferencia entre regularizar y promocionar la materia?',
        'expected_keywords': ['4', '6', 'trabajos prácticos', 'parcial', '80', 'asistencia'],
        'expected_docs': ['doc_evaluacion'],
    },
    # ── Edge case ────────────────────────────────────────────────
    {
        'id': 'edge-1', 'tipo': 'edge-case',
        'pregunta': '¿Cuándo es la entrega del TP3?',
        'expected_keywords': ['no', 'sólo', 'TP1', 'TP2', 'dos'],
        'expected_doc': None,  # NO hay TP3 — el sistema debe abstenerse
    },
]

print(f'✓ Benchmark con {len(BENCHMARK)} queries')
for q in BENCHMARK:
    print(f'  [{q["tipo"]:18}] {q["pregunta"][:60]}')


## 8. ✏️ Ejercicio — side-by-side LLM puro vs RAG naive (10 min)

Corremos las 7 queries en **paralelo** por LLM puro y RAG. La métrica `score_keywords` cuenta cuántos de los hechos específicos del corpus aparecen en cada respuesta. **El LLM puro va a sacar casi 0 en los datos específicos** porque no puede saberlos.


In [ ]:
import pandas as pd


def side_by_side(benchmark, n_chunks=3):
    """Corre cada query por LLM puro y RAG naive."""
    rows = []
    for q in benchmark:
        resp_llm = llamar_llm([
            {'role': 'system', 'content': 'Respondé en español, máximo 4 oraciones.'},
            {'role': 'user', 'content': q['pregunta']},
        ], temperature=0.3)
        score_llm = score_keywords(resp_llm, q['expected_keywords'])

        resp_rag, results = rag_query(q['pregunta'], n_chunks=n_chunks, verbose=False)
        score_rag = score_keywords(resp_rag, q['expected_keywords'])
        docs_rag = [m['doc_id'] for m in results['metadatas'][0]]
        doc_ok = (q.get('expected_doc') in docs_rag) if q.get('expected_doc') else None

        rows.append({
            'id': q['id'],
            'tipo': q['tipo'],
            'pregunta': q['pregunta'][:45] + '...',
            'llm_kw': f'{score_llm}/{len(q["expected_keywords"])}',
            'rag_kw': f'{score_rag}/{len(q["expected_keywords"])}',
            'rag_doc_ok': '✓' if doc_ok else ('✗' if doc_ok is False else 'n/a'),
            'resp_llm': resp_llm,
            'resp_rag': resp_rag,
        })
    return pd.DataFrame(rows)


print('Corriendo benchmark (puede tardar 1-2 min)...\n')
df_sxs = side_by_side(BENCHMARK)
print(df_sxs[['id', 'tipo', 'pregunta', 'llm_kw', 'rag_kw', 'rag_doc_ok']].to_string(index=False))


### Inspección detallada — ¿qué inventó el LLM puro?

Mirá las respuestas. **Buscá fechas, nombres y umbrales** en la del LLM puro: si no están en los hechos correctos del corpus, **son inventos**.


In [ ]:
for _, row in df_sxs.iterrows():
    print('═' * 70)
    print(f'[{row["tipo"]}] {row["id"]} — {row["pregunta"]}')
    print(f'\n  LLM puro ({row["llm_kw"]}):')
    print(f'    {row["resp_llm"][:300]}')
    print(f'\n  RAG naive ({row["rag_kw"]}, doc_ok={row["rag_doc_ok"]}):')
    print(f'    {row["resp_rag"][:300]}')
    print()


## 9. Visualización — el contraste LLM puro vs RAG naive


In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def to_frac(s):
    n, m = s.split('/')
    return int(n) / int(m) if int(m) > 0 else 0


df_sxs['llm_frac'] = df_sxs['llm_kw'].apply(to_frac)
df_sxs['rag_frac'] = df_sxs['rag_kw'].apply(to_frac)

by_tipo = df_sxs.groupby('tipo')[['llm_frac', 'rag_frac']].mean()

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(by_tipo))
w = 0.35
ax.bar(x - w/2, by_tipo['llm_frac'], w, label='LLM puro', color='#E07474')
ax.bar(x + w/2, by_tipo['rag_frac'], w, label='RAG naive', color='#5BC85B')
ax.set_xticks(x)
ax.set_xticklabels(by_tipo.index, rotation=20, ha='right')
ax.set_ylabel('Fracción de keywords cubiertos (avg)')
ax.set_title('LLM puro vs RAG naive — score promedio por tipo de query')
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(axis='y', alpha=0.3)
for i, (lv, rv) in enumerate(zip(by_tipo['llm_frac'], by_tipo['rag_frac'])):
    ax.text(i - w/2, lv + 0.02, f'{lv:.0%}', ha='center', fontsize=9)
    ax.text(i + w/2, rv + 0.02, f'{rv:.0%}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

print()
print(f'Score total LLM puro:  {df_sxs["llm_frac"].mean():.0%}')
print(f'Score total RAG naive: {df_sxs["rag_frac"].mean():.0%}')


### Reflexión rápida

1. **El LLM puro no puede saber datos específicos** (fechas, cargos, ordenanzas). Si menciona alguno, lo inventa.
2. **RAG naive (dense only) ya gana en datos específicos**, pero todavía falla en algunos casos:
   - Queries léxicas con siglas o términos exactos (CIDISI, Ordenanza 1877) donde dense se confunde con vecinos semánticos.
   - **Multi-hop** (combinar información de 2 documentos).
   - Queries donde el chunk correcto queda en la "cola" del ranking.
3. En la Parte 4 vamos a subir el nivel usando **LangChain** — un framework que ya resuelve estos casos con técnicas de producción: **hybrid retrieval** (BM25 + dense fusionados con Reciprocal Rank Fusion) y **reranking** (cross-encoder como segundo paso).


# Parte 4 — Subiendo a un framework: LangChain

Hasta acá implementamos RAG **a mano** para entender qué pasa adentro. Para construir cosas reales, vas a usar un framework que ya resolvió los problemas que ahora te quedan: hybrid retrieval que funcione, reranking, switching entre vectorstores, manejo de documentos.

Vamos a recrear el pipeline con **LangChain** y mostrar que:

1. **Hybrid retrieval que funciona** — `EnsembleRetriever` usa Reciprocal Rank Fusion (RRF), no weighted sum. Esto evita el modo de falla que ya tenés en la cabeza.
2. **Reranker plug-and-play** — agregás 5 líneas con un cross-encoder y mejorás los casos donde el retriever pone la respuesta correcta en la cola.
3. **Menos código, abstracciones limpias** — todo orbitando un único contrato: `retriever.invoke(query) -> list[Document]`.

> 💡 **Por qué no `langchain-experimental` con HyDE o ParentDocumentRetriever?** Son técnicas valiosas, pero queremos mostrar el **80% del valor con la mínima sorpresa**: hybrid + rerank. El resto está documentado en LangChain y lo aplicás cuando lo necesites.


In [ ]:
# Instalamos LangChain + sub-paquetes. Colab suele tener langchain 0.x preinstalado;
# usamos --upgrade para forzar 1.x (que es donde vive langchain-huggingface como paquete propio).
!pip install -q --upgrade langchain langchain-community langchain-classic langchain-chroma langchain-huggingface


In [ ]:
# Verificación — si era la primera vez que instalabas estos paquetes en este runtime,
# Python tiene cacheada la versión vieja y esta celda va a fallar con ModuleNotFoundError.
# Solución: Runtime → Restart session (o Ctrl+M .) y volvé a correr desde la celda anterior.
try:
    import langchain_huggingface, langchain_chroma, langchain_classic, langchain_community  # noqa: F401
    print('✓ LangChain y sub-paquetes listos.')
except ModuleNotFoundError as e:
    print(f'⚠️  {e}')
    print('   → Reiniciá el runtime (Runtime → Restart session) y corré las celdas de nuevo.')


## 10. Reconstruir el pipeline en LangChain

LangChain modela todo con tipos consistentes:

| Concepto | Tipo LC |
|---|---|
| Un chunk | `Document(page_content, metadata)` |
| Modelo de embeddings | `Embeddings` (wrapper sobre HuggingFace, OpenAI, ...) |
| Almacén vectorial | `VectorStore` (Chroma, Pinecone, Weaviate, ...) |
| Búsqueda | `Retriever` con `.invoke(query) -> list[Document]` |

Convertimos nuestros `all_chunks` + `all_metadatas` a `Document`s, construimos un Chroma con los mismos embeddings de antes, y armamos un retriever BM25 standalone.


In [ ]:
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

# 1) Documents = chunks + metadatos en una estructura uniforme
lc_docs = [
    Document(page_content=chunk, metadata={'doc_id': meta['doc_id'], 'titulo': meta['titulo']})
    for chunk, meta in zip(all_chunks, all_metadatas)
]
print(f'✓ {len(lc_docs)} Documents')

# 2) Embeddings (el MISMO modelo de la Parte 2, ahora wrappeado por LangChain)
lc_emb = HuggingFaceEmbeddings(model_name='paraphrase-multilingual-MiniLM-L12-v2')

# 3) Vectorstore Chroma (nueva collection, separada de la que usamos a mano)
lc_vs = Chroma.from_documents(lc_docs, lc_emb, collection_name='clase3_lc')
lc_dense = lc_vs.as_retriever(search_kwargs={'k': 3})

# 4) BM25 retriever standalone
lc_bm25 = BM25Retriever.from_documents(lc_docs)
lc_bm25.k = 3

# 5) Hybrid con RRF (es el default de EnsembleRetriever)
lc_hybrid = EnsembleRetriever(retrievers=[lc_bm25, lc_dense], weights=[0.5, 0.5])

print('✓ lc_dense, lc_bm25 y lc_hybrid listos')


## 11. ¿Por qué RRF no tiene el bug del weighted-sum?

Si vinieras de implementar hybrid retrieval a mano (que es lo que NO hicimos acá), te toparías con un modo de falla clásico: cuando BM25 dice "el chunk correcto está en el #1" pero dense lo manda al #5, **un promedio ponderado lo expulsa fuera del top-k**.

**Reciprocal Rank Fusion (RRF)** resuelve esto sumando recíprocos del ranking, no scores:

$$\text{score}(d) = \sum_{i \in \text{retrievers}} \frac{1}{k + \text{rank}_i(d)}$$

Con $k=60$. Si un chunk aparece en el top-3 de **cualquier** retriever, gana un score competitivo, sin importar el valor crudo del otro. Es el método estándar en producción (Elasticsearch, OpenSearch, Vespa).

Probemos con una query léxica difícil — "¿Quiénes son los docentes?". El chunk correcto contiene "Profesora Titular Milagros Gutiérrez y el Profesor Adjunto Jorge Roa". BM25 lo encuentra (palabras exactas: "docentes", "Titular", "Adjunto"); dense se distrae con el chunk de bibliografía (que tiene "Inteligencia Artificial" muchas veces).


In [ ]:
query = '¿Quiénes son los docentes de la cátedra Inteligencia Artificial?'
print(f'Query: "{query}"\n')

print('── lc_bm25 (k=3): ──')
for i, d in enumerate(lc_bm25.invoke(query), 1):
    print(f'  #{i} [{d.metadata["doc_id"]:18}] "{d.page_content[:75]}"')

print('\n── lc_dense (k=3): ──')
for i, d in enumerate(lc_dense.invoke(query), 1):
    print(f'  #{i} [{d.metadata["doc_id"]:18}] "{d.page_content[:75]}"')

print('\n── lc_hybrid con RRF (top 3): ──')
for i, d in enumerate(lc_hybrid.invoke(query)[:3], 1):
    print(f'  #{i} [{d.metadata["doc_id"]:18}] "{d.page_content[:75]}"')

print('\n💡 RRF mantiene el chunk con los nombres de los docentes en el top-3,')
print('   aunque dense lo había dejado afuera. Eso es lo que un weighted-sum naive')
print('   no garantiza.')


## 12. Reranking — el último filtro

Hasta acá todo nuestro pipeline es **bi-encoder**: los embeddings de query y documentos se calculan **por separado** y después se comparan. Es rápido (podés indexar millones de docs offline), pero impreciso — la query nunca "ve" al doc.

Un **cross-encoder** procesa el par `(query, doc)` **juntos** por una red neuronal. Mucho más preciso, pero también más lento — no se puede precalcular. Por eso se usa como **segundo paso** del pipeline:

1. **Bi-encoder** (dense + BM25 + RRF) → trae top 6-10 candidatos rápidos.
2. **Cross-encoder** (rerank) → reordena y devuelve top 3.

Usamos `BAAI/bge-reranker-base` (multilingual, ~280 MB).


In [ ]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

# Para reranking, subimos k a 6 en cada retriever base — el cross-encoder necesita material para elegir
lc_bm25_rk = BM25Retriever.from_documents(lc_docs)
lc_bm25_rk.k = 6
lc_dense_rk = lc_vs.as_retriever(search_kwargs={'k': 6})
lc_hybrid_rk = EnsembleRetriever(retrievers=[lc_bm25_rk, lc_dense_rk], weights=[0.5, 0.5])

# Cross-encoder reranker (multilingual)
ce_model = HuggingFaceCrossEncoder(model_name='BAAI/bge-reranker-base')
reranker = CrossEncoderReranker(model=ce_model, top_n=3)

# Pipeline final: hybrid (k=6 cada uno) → rerank → top 3
lc_hybrid_rerank = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=lc_hybrid_rk,
)

print('✓ lc_hybrid_rerank listo')
print('\nProbemos con la query rota antes — "¿Cómo está organizado el sistema de evaluación?":')
query = '¿Cómo está organizado el sistema de evaluación de la materia?'
for i, d in enumerate(lc_hybrid_rerank.invoke(query), 1):
    print(f'  #{i} [{d.metadata["doc_id"]:18}] "{d.page_content[:75]}"')
print('\n💡 El reranker trae varios chunks de doc_evaluacion al top-3 — incluyendo el de')
print('   "Para regularizar el cursado..." que es la respuesta directa. El hybrid solo')
print('   no lo lograba.')


## 13. RAG end-to-end con LangChain

Wrappeamos un `rag_query_lc(query, retriever)` que toma **cualquier** retriever LC y devuelve respuesta + sources. La misma función va a servir para correr el benchmark con `lc_dense`, `lc_hybrid` o `lc_hybrid_rerank` sin duplicar código — exactamente el valor que da la abstracción `Retriever`.


In [ ]:
def rag_query_lc(query, retriever, verbose=False):
    """RAG query usando un retriever de LangChain."""
    docs = retriever.invoke(query)
    contexto = '\n\n'.join(f'[{d.metadata.get("doc_id", "?")}] {d.page_content}' for d in docs)

    resp = llamar_llm([
        {'role': 'system', 'content':
            'Sos un asistente de la cátedra de Inteligencia Artificial de UTN-FRSF. '
            'Respondé la pregunta usando ÚNICAMENTE el contexto provisto. '
            'Sé concreto, citá los datos exactos, máximo 5 oraciones. '
            'Si el contexto no alcanza para responder, decilo brevemente.'},
        {'role': 'user', 'content': f'Contexto:\n{contexto}\n\nPregunta: {query}'},
    ], temperature=0.3)

    sources = [d.metadata.get('doc_id') for d in docs]
    if verbose:
        print(f'[sources] {sources}')
        print(f'[resp]\n{resp}')
    return {'response': resp, 'sources': sources}


# Demo rápida
r = rag_query_lc(
    '¿Quiénes son los docentes de la cátedra Inteligencia Artificial?',
    lc_hybrid_rerank,
    verbose=True,
)


## 14. Comparación cualitativa — 3 casos concretos

En lugar de correr el benchmark completo y reportar un porcentaje, vamos a inspeccionar **3 queries específicas** comparando las respuestas de los 3 retrievers: `rag_query` naive (dense only), `lc_hybrid` (BM25 + dense + RRF) y `lc_hybrid_rerank` (+ cross-encoder).

**¿Por qué cualitativo y no aggregate score?** La métrica que usamos en la Parte 3 — `score_keywords` — cuenta presencia literal de palabras-clave esperadas. Es buena para detectar si el LLM puro inventa datos vs si el RAG los trae del corpus (allá los datos están o no están). Pero **está mal calibrada para comparar técnicas de retrieval avanzadas**: un retriever puede traer el chunk *semánticamente correcto* y aún así perder en keyword coverage si el LLM redacta la respuesta con otras palabras.

> La evaluación seria de un sistema RAG — *LLM-as-judge*, *RAGAS*, métricas de relevancia con anotación humana — la cubrimos en **Clase 3b**. Por ahora nos quedamos con inspección directa.

Los 3 casos están elegidos para mostrar 3 patrones distintos:

| Caso | Query | Lo que esperamos ver |
|---|---|---|
| **1** | ¿Quiénes son los docentes? | Hybrid gana fuerte sobre naive (query léxica). |
| **2** | ¿Cómo está organizado el sistema de evaluación? | Reranker mejora sobre hybrid (ambigüedad semántica). |
| **3** | ¿Cuál es la diferencia entre regularizar y promocionar? | Reranker introduce un tradeoff visible. |

In [ ]:
def comparar_caso(titulo, query):
    """Corre los 3 retrievers para una query y muestra el side-by-side."""
    print('═' * 90)
    print(f'  {titulo}')
    print('═' * 90)
    print(f'Query: "{query}"\n')

    # 1) Naive (dense only — la pipeline de la Parte 2)
    resp_naive, results_naive = rag_query(query, n_chunks=3, verbose=False)
    sources_naive = [m['doc_id'] for m in results_naive['metadatas'][0]]

    # 2) LangChain hybrid (BM25 + dense + RRF)
    r_hyb = rag_query_lc(query, lc_hybrid)

    # 3) LangChain hybrid + cross-encoder reranker
    r_rk = rag_query_lc(query, lc_hybrid_rerank)

    for label, sources, resp in [
        ('naive (dense only)            ', sources_naive, resp_naive),
        ('lc_hybrid (RRF)               ', r_hyb['sources'], r_hyb['response']),
        ('lc_hybrid_rerank (+ cross-enc)', r_rk['sources'], r_rk['response']),
    ]:
        print(f'── {label} ──')
        print(f'   sources: {sources}')
        print(f'   resp:')
        for line in resp.split('\n'):
            print(f'     {line}')
        print()

In [ ]:
# CASO 1 — Query léxica
# Esperamos que BM25 capture "docentes" exacto y RRF lo preserve.
# Dense suele desviarse al chunk de bibliografía (que tiene "Inteligencia Artificial" 4 veces).
comparar_caso(
    'CASO 1 — Query léxica: ¿quiénes son los docentes?',
    '¿Quiénes son los docentes de la cátedra Inteligencia Artificial?'
)

print('💡 Qué mirar:')
print('   El chunk con "Profesora Titular Milagros Gutiérrez y Profesor Adjunto Jorge Roa"')
print('   vive en doc_identidad. Cualquier respuesta que NO mencione esos nombres está')
print('   adivinando porque su retriever no trajo el chunk al top-3.')

In [ ]:
# CASO 2 — Ambigüedad semántica
# La palabra "evaluación" del query matchea con dos cosas en el corpus:
#  (a) "aspectos a evaluar en cada TP" (doc_tps — sobre criterios de calificación de TPs)
#  (b) "Para regularizar el cursado..." (doc_evaluacion — el régimen académico real)
# El alumno espera (b). Veamos quién lo trae.
comparar_caso(
    'CASO 2 — Ambigüedad semántica: ¿cómo se evalúa la materia?',
    '¿Cómo está organizado el sistema de evaluación de la materia?'
)

print('💡 Qué mirar:')
print('   Si la respuesta menciona "regularizar / promocionar / puntajes ≥4 o ≥6 /')
print('   80% asistencia" → el retriever trajo doc_evaluacion (correcto).')
print('   Si sólo habla de "informe técnico, presentación, alcance de objetivos" →')
print('   trajo doc_tps (sobre rúbrica de TPs, no sobre el régimen de aprobación).')

In [ ]:
# CASO 3 — El tradeoff del reranker
# Query: diferencia entre regularizar y promocionar. La respuesta correcta requiere
# mencionar los umbrales numéricos (≥4 para regularizar, ≥6 para promocionar) y los
# requisitos comunes (TPs, parcial, 80% asistencia, debate).
# Veamos si los 3 retrievers traen el chunk con esos umbrales — y observá que el rerank
# no siempre es la respuesta "más completa" según presencia literal de números.
comparar_caso(
    'CASO 3 — Tradeoff del reranker: ¿regularizar vs promocionar?',
    '¿Cuál es la diferencia entre regularizar y promocionar la materia?'
)

print('💡 Qué mirar:')
print('   Las respuestas que incluyen los números literales "4" y "6", y palabras como')
print('   "trabajos prácticos", "parcial", "asistencia" cubren los hechos importantes.')
print('   El reranker a veces trae chunks semánticamente relevantes pero que no contienen')
print('   esos números literales, dando una respuesta que "habla del tema" pero pierde')
print('   precisión. Este es el motivo por el cual no podés cerrar la decisión sobre')
print('   reranker mirando keyword coverage — necesitás medir relevancia (Clase 3b).')

## 15. Cierre — qué te llevás de la Parte 4

### Cuándo subir a un framework

| Etapa | Recomendación |
|---|---|
| Prototipo, *"¿esto va a funcionar?"* | Hand-roll (como Partes 1-3) para entender qué pasa adentro. |
| Producto interno, MVP | LangChain / LlamaIndex / Haystack — Parte 4. |
| Producto a escala con SLA | Framework + componentes custom para lo crítico (retrieval, eval, observability). |

### Qué te llevaste hoy

- **Cómo funciona RAG por dentro** — chunking, embeddings, retrieval, augmentation.
- **Hybrid (BM25 + dense) es mejor que solo dense**, y la fusión correcta importa: **RRF** resuelve el modo de falla del weighted-sum.
- **Reranking ayuda en algunas queries pero introduce tradeoffs reales** — no lo defaulteás sin medir en tu dominio.
- **Una abstracción uniforme** (`Retriever`) que te permite cambiar implementaciones sin tocar el resto del código.

### Decisión honesta sobre reranker

- **Defaulteá** hybrid (BM25 + dense + RRF). Es ganancia limpia, baja fricción, hacelo siempre.
- **Probá** reranker en tu corpus y **mide** con LLM-as-judge o anotación humana (Clase 3b). Si gana, defaulteá; si no, ahorrate la latencia y el peso del modelo.

### Siguientes pasos (más allá de la clase de hoy)

- **HyDE** — generar un documento hipotético antes del embedding (LangChain `HypotheticalDocumentEmbedder`).
- **Parent-child chunking** — chunks chicos para retrieval, grandes para el LLM (`ParentDocumentRetriever`).
- **Graph RAG** — embedding retrieval combinado con grafos de conocimiento.
- **Evaluación sistemática** — **Clase 3b**: cómo medir si tu RAG está mejorando (LLM-as-judge, RAGAS, observability con Arize AX).